<a href="https://colab.research.google.com/github/biopharma26/PracticeNotebooks/blob/main/p304_MD_Trajectory_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Molecular Dynamics: M-Loop Stabilization Analysis

**Section mirrored:** Report §4 (MD trajectories, Figure 4: RMSD / RMSF / Rg)

**Reality check:** genuine 100 ns all-atom MD of a tubulin–taxane complex requires GROMACS/AMBER/OpenMM,
a solvated + ionized system, minimization, NVT/NPT equilibration, and hours-to-days of GPU time — it cannot
be run inside a quick notebook check. What's provided here:

1. A **real, runnable OpenMM protocol** (Section 2) you can launch in Colab (with a GPU runtime) against an
   actual prepared tubulin–taxane system.
2. A **real MDAnalysis-based RMSD/RMSF/Rg analysis routine** (Section 3) that operates on whatever
   trajectory file you produce — from OpenMM, GROMACS, or Desmond.
3. A clearly-labeled **illustrative placeholder trajectory** (Section 4) only so the notebook produces a
   Figure-4-style dashboard end-to-end before you have real simulation output.

> ⚠️ Section 4's curves are synthetic and exist only to demonstrate the analysis code — they are not a
> simulation result and must not be reported as one.


In [ ]:
!pip -q install MDAnalysis
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120


## 1. System preparation (outline)

1. Take the docked tubulin–taxane complex from Notebook 03 (or a deposited co-crystal structure).
2. Solvate in a TIP3P water box with 0.15 M NaCl, using e.g. `pdbfixer` + `openmm.app.Modeller`.
3. Parameterize the ligand with a small-molecule force field (GAFF2/OpenFF) since taxanes aren't covered
   by standard protein force fields.
4. Minimize, then equilibrate under NVT then NPT before production.

In [ ]:
openmm_scaffold = r"""
# Run in a Colab GPU runtime with OpenMM installed — not executed in this notebook build.

!pip -q install openmm pdbfixer

from openmm.app import *
from openmm import *
from openmm.unit import nanometer, kelvin, picosecond, picoseconds

# 1. Load prepared, solvated system (from receptor+ligand prep in Notebook 03 + a water box)
pdb = PDBFile("solvated_complex.pdb")
forcefield = ForceField("amber14-all.xml", "amber14/tip3pfb.xml")  # + ligand force field, e.g. GAFF2/OpenFF

system = forcefield.createSystem(pdb.topology, nonbondedMethod=PME,
                                  nonbondedCutoff=1.0*nanometer, constraints=HBonds)
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)

# 2. Minimize
simulation.minimizeEnergy()

# 3. Equilibrate (NVT then NPT) — add a Barostat for NPT
simulation.context.setVelocitiesToTemperature(300*kelvin)
simulation.step(50000)  # ~100 ps equilibration, extend for production

# 4. Production run with trajectory reporting
simulation.reporters.append(DCDReporter("trajectory.dcd", 5000))  # every 10 ps at 2 fs step
simulation.reporters.append(StateDataReporter("md_log.csv", 5000, step=True,
                                               potentialEnergy=True, temperature=True))
simulation.step(50_000_000)  # 100 ns at 2 fs/step
"""
print(openmm_scaffold)


## 2. Real trajectory analysis (MDAnalysis)

Point this at whatever trajectory/topology you actually generate — this code is genuine and will compute
real RMSD, per-residue RMSF, and radius of gyration once given real files.

In [ ]:
analysis_scaffold = r"""
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align

u = mda.Universe("topology.pdb", "trajectory.dcd")
protein = u.select_atoms("protein and name CA")

# Align all frames to the first frame on the protein backbone before RMSD/RMSF
aligner = align.AlignTraj(u, u, select="protein and name CA", in_memory=True)
aligner.run()

# RMSD over time
R = rms.RMSD(u, u, select="protein and name CA", ref_frame=0)
R.run()
rmsd_series = R.results.rmsd[:, 2]  # column 2 = RMSD in Angstrom

# Per-residue RMSF (M-loop = residues 270-285 per the report)
rmsf = rms.RMSF(protein).run()
rmsf_by_residue = rmsf.results.rmsf

# Radius of gyration over time
rg_series = [protein.radius_of_gyration() for ts in u.trajectory]
"""
print(analysis_scaffold)


## 3. Illustrative placeholder dashboard

**Synthetic curves only** — reproduces the *layout* of report Figure 4 so you can sanity-check the plotting
code before wiring in real MDAnalysis output above.

In [ ]:
np.random.seed(3)
t = np.linspace(0, 100, 500)  # ns

def rmsd_curve(plateau, noise, rise_rate):
    return plateau * (1 - np.exp(-rise_rate * t)) + np.random.normal(0, noise, len(t))

rmsd_apo = rmsd_curve(2.6, 0.15, 0.08)
rmsd_paclitaxel = rmsd_curve(1.8, 0.10, 0.10)
rmsd_cabazitaxel = rmsd_curve(1.2, 0.08, 0.12)

residue_idx = np.arange(1, 431)
def rmsf_profile(mloop_suppression):
    base = 0.8 + 0.5 * np.sin(residue_idx / 15) ** 2 + np.random.normal(0, 0.15, len(residue_idx))
    mloop = (residue_idx >= 270) & (residue_idx <= 285)
    base[mloop] = base[mloop] * mloop_suppression + np.random.normal(0, 0.05, mloop.sum())
    return np.clip(base, 0, None)

rmsf_apo = rmsf_profile(1.0) + 1.0
rmsf_paclitaxel = rmsf_profile(0.5)
rmsf_cabazitaxel = rmsf_profile(0.2)

rg_apo = 20.8 + np.random.normal(0, 0.08, len(t))
rg_paclitaxel = 20.4 + np.random.normal(0, 0.04, len(t))
rg_cabazitaxel = 20.35 + np.random.normal(0, 0.03, len(t))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(t, rmsd_apo, color="lightgray", label="Apo Beta-Tubulin")
axes[0].plot(t, rmsd_paclitaxel, color="tab:blue", label="Paclitaxel-Tubulin")
axes[0].plot(t, rmsd_cabazitaxel, color="tab:red", label="Cabazitaxel-Tubulin")
axes[0].set_xlabel("Simulation Time (ns)"); axes[0].set_ylabel("Backbone RMSD (Å)")
axes[0].set_title("Root Mean Square Deviation (RMSD)"); axes[0].legend(fontsize=7)

axes[1].plot(residue_idx, rmsf_apo, color="lightgray", label="Apo Beta-Tubulin")
axes[1].plot(residue_idx, rmsf_paclitaxel, color="tab:blue", label="Paclitaxel-Bound")
axes[1].plot(residue_idx, rmsf_cabazitaxel, color="tab:red", label="Cabazitaxel-Bound")
axes[1].axvspan(270, 285, color="pink", alpha=0.3, label="M-loop (270-285)")
axes[1].set_xlabel("Beta-Tubulin Residue Index"); axes[1].set_ylabel("RMSF (Å)")
axes[1].set_title("Root Mean Square Fluctuation (RMSF)"); axes[1].legend(fontsize=7)

axes[2].plot(t, rg_apo, color="lightgray", label="Apo Beta-Tubulin")
axes[2].plot(t, rg_paclitaxel, color="tab:blue", label="Paclitaxel-Tubulin")
axes[2].plot(t, rg_cabazitaxel, color="tab:red", label="Cabazitaxel-Tubulin")
axes[2].set_xlabel("Simulation Time (ns)"); axes[2].set_ylabel("Radius of Gyration (Å)")
axes[2].set_title("Radius of Gyration (Rg)"); axes[2].legend(fontsize=7)

fig.suptitle("ILLUSTRATIVE PLACEHOLDER — not a simulation result", fontsize=10, color="firebrick")
plt.tight_layout()
plt.savefig("figure4_md_dashboard_PLACEHOLDER.png", dpi=300)
plt.show()


---
### Next steps for a real manuscript
- Run the OpenMM scaffold (Section 1) on an actual prepared, solvated tubulin–taxane system for a real
  production trajectory.
- Feed the resulting `.dcd`/`.xtc` into the MDAnalysis code in Section 2 to get genuine RMSD/RMSF/Rg values.
- Only then replace the illustrative Section 3 dashboard — never report the placeholder curves as findings.
